# Day 5 - S3 from pandas

Read straight from the bucket, convert CSV to Parquet, write back, share with a presigned URL.

Everything here uses the `ds` CLI profile from Day 3. No keys in this file.

In [2]:
import time

import boto3
import pandas as pd

PROFILE = "ds"
BUCKET = "dave-ds-lab-ca"

# pandas -> s3fs -> boto. storage_options is how you tell s3fs which profile to use.
S3 = {"profile": PROFILE}

session = boto3.Session(profile_name=PROFILE)
s3 = session.client("s3")

def size_mb(key):
    """Object size in MB without downloading it (HEAD request)."""
    return s3.head_object(Bucket=BUCKET, Key=key)["ContentLength"] / 1e6

print("Identity:", session.client("sts").get_caller_identity()["Arn"])

Identity: arn:aws:iam::754928766714:user/D@ve_Admin


## 1. Read Parquet directly from S3

No download step. `s3://` is just another path to pandas once `s3fs` is installed.

In [ ]:
PQ_RAW = f"s3://{BUCKET}/raw/yellow_tripdata_2024-01.parquet"

t0 = time.perf_counter()
df = pd.read_parquet(PQ_RAW, storage_options=S3)
t_parquet_read = time.perf_counter() - t0

print(f"{len(df):,} rows x {df.shape[1]} cols in {t_parquet_read:.1f}s")
df.head()

In [ ]:
df.dtypes

## 2. Write a CSV copy to `raw/`

We need a CSV in the bucket to compare against. Real life is usually the other way round - you
receive CSV and convert it - but the comparison is the same. This takes a few minutes: it is
~300 MB going up. (Rerun-safe: skips if it already exists.)

In [ ]:
CSV_KEY = "raw/yellow_tripdata_2024-01.csv"
CSV_RAW = f"s3://{BUCKET}/{CSV_KEY}"

try:
    s3.head_object(Bucket=BUCKET, Key=CSV_KEY)
    print("CSV already in S3, skipping write")
except s3.exceptions.ClientError:
    t0 = time.perf_counter()
    df.to_csv(CSV_RAW, index=False, storage_options=S3)
    print(f"wrote CSV in {time.perf_counter() - t0:.0f}s")

print(f"CSV size: {size_mb(CSV_KEY):.0f} MB")

## 3. Read the CSV back, convert to Parquet (Snappy), write to `processed/`

Watch two things: how long the CSV read takes vs the Parquet read above, and what happens to the
dtypes (CSV has no schema - timestamps come back as strings).

In [ ]:
t0 = time.perf_counter()
df_csv = pd.read_csv(CSV_RAW, storage_options=S3)
t_csv_read = time.perf_counter() - t0
print(f"CSV read: {t_csv_read:.1f}s  (Parquet read was {t_parquet_read:.1f}s)")

# CSV lost the timestamp types - put them back before writing Parquet.
for col in ["tpep_pickup_datetime", "tpep_dropoff_datetime"]:
    df_csv[col] = pd.to_datetime(df_csv[col])

df_csv.dtypes

In [ ]:
PQ_KEY = "processed/yellow_tripdata_2024-01.parquet"
PQ_PROCESSED = f"s3://{BUCKET}/{PQ_KEY}"

t0 = time.perf_counter()
df_csv.to_parquet(PQ_PROCESSED, compression="snappy", index=False, storage_options=S3)
print(f"wrote Parquet in {time.perf_counter() - t0:.0f}s")

## 4. The size comparison

This is the number to remember. Same rows, same columns.

In [ ]:
csv_mb = size_mb(CSV_KEY)
pq_mb = size_mb(PQ_KEY)

print(f"CSV      raw/        {csv_mb:7.1f} MB   read {t_csv_read:5.1f}s")
print(f"Parquet  processed/  {pq_mb:7.1f} MB   read {t_parquet_read:5.1f}s")
print(f"\nParquet is {csv_mb / pq_mb:.1f}x smaller.")
print("On Athena (Day 9) that ratio is also roughly the cost ratio - it bills per byte scanned.")

## 5. Presigned URL - share a private object without making it public

The URL carries a signature made with *your* credentials and expires. Anyone holding it can
download that one object until then. The bucket stays private.

In [ ]:
url = s3.generate_presigned_url(
    "get_object",
    Params={"Bucket": BUCKET, "Key": PQ_KEY},
    ExpiresIn=600,  # seconds
)
print(url)
print("\nPaste this in an incognito window: it downloads. Wait 10 minutes and try again: AccessDenied.")

## 6. What is in the bucket now

In [ ]:
resp = s3.list_objects_v2(Bucket=BUCKET)
for obj in resp.get("Contents", []):
    if obj["Size"]:
        print(f"{obj['Size'] / 1e6:8.1f} MB  {obj['Key']}")